In [53]:
%reset -f
import arcpy
arcpy.ResetEnvironments()
import os

In [54]:
# ── PATHS ────────────────────────────────────────────────────────────────────────────────────────────

# Input USC and no-overlap GBC bathy + slope rasters
gbc_bathy = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_thesisextent.gdb\nooverlap_GBC15a_bathy_p26920_v6642"
usc_bathy = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\Jeremiah_NOAA_10meter_USCaribbean.gdb\USC10m_bathy_p26920_v6642"
gbc_slope = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_thesisextent.gdb\nooverlap_GBC15a_slope_p26920_v6642"
usc_slope = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\Jeremiah_NOAA_10meter_USCaribbean.gdb\USC10m_slope_p26920_v6642"

# Geodatabase for output mosaic datasets
gdb = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\MASTER_RASTERS_USC_GBC_MERGED.gdb"

# Output final standalone merged bathy + slope rasters
final_bathy_raster = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\MASTER_RASTERS_USC_GBC_MERGED.gdb\BATHY_USCGBC_p26920_v6642"
final_slope_raster = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\MASTER_RASTERS_USC_GBC_MERGED.gdb\SLOPE_USCGBC_p26920_v6642"

# Output TIFF files of final merged bathy + slope rasters
bathy_tif = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\exported_files\BATHY_USCGBC_p26920_v6642.tif"
slope_tif = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\exported_files\SLOPE_USCGBC_p26920_v6642.tif"

In [55]:
# ── SETTINGS ─────────────────────────────────────────────────────────────────────────────────────────
arcpy.env.overwriteOutput = True    # overwrite existing outputs on rerun
arcpy.env.compression = "NONE"      # set compression to NONE for output mosaics and rasters

# set environment coordinate system: horizontal EPSG:26920 + vertical EPSG:6642 (coord system set within Create Mosaic Dataset tool *should* be carried over by Copy Raster tool from the input mosaic, but set explicitly for entire environment just to be safe and ensure output merged rasters include both horizontal + vertical coord systems)
arcpy.env.outputCoordinateSystem = arcpy.SpatialReference(26920, 6642)

In [56]:
# ── INSPECT INPUT RASTERS ────────────────────────────────────────────────────────────────────────────

# Function to print all info (number of bands, pixel type, min, max, cell size, coordinate systems) for each raster
def print_raster_info(raster):

    # get number of bands info
    d = arcpy.Describe(raster)
    # get pixel type/min/max/cell size info (r = arcpy.Raster() loads raster dataset into a Raster object so arcpy can access stats)
    r = arcpy.Raster(raster)
    # get spatial reference info
    sr = d.spatialReference
    vcs = sr.VCS.name if sr.VCS else "NONE"

    # info print block for raster
    print(f"\n---------------- {os.path.basename(raster)} ----------------: \n")
    print(f"   Number bands = {d.bandCount}")
    print(f"   Pixel type = {r.pixelType}")
    print(f"   ⬇️ Min = {r.minimum}")
    print(f"   ⬆️ Max = {r.maximum}")
    print(f"   📐 Cell size = {r.meanCellWidth}")
    print(f"   Horizontal SR: {sr.name}")
    print(f"   Vertical SR: {vcs}")
    print(f"\n")


# Calculate Statistics (Data Management Tools): (default X/Y skip factor = 1; skip rasters with existing stats calculated)
for stats_path1 in [gbc_bathy, usc_bathy, gbc_slope, usc_slope]:
    arcpy.management.CalculateStatistics(
        in_raster_dataset=stats_path1,
        skip_existing="SKIP_EXISTING"
    )

# Call function to print info for each input raster
print_raster_info(gbc_bathy)
print_raster_info(usc_bathy)
print_raster_info(gbc_slope)
print_raster_info(usc_slope)


---------------- nooverlap_GBC15a_bathy_p26920_v6642 ----------------: 

   Number bands = 1
   Pixel type = S16
   ⬇️ Min = -8619.0
   ⬆️ Max = 0.0
   📐 Cell size = 472.8452741843477
   Horizontal SR: NAD_1983_UTM_Zone_20N
   Vertical SR: VIVD09_height



---------------- USC10m_bathy_p26920_v6642 ----------------: 

   Number bands = 1
   Pixel type = F32
   ⬇️ Min = -6239.16943359375
   ⬆️ Max = 0.0
   📐 Cell size = 10.0
   Horizontal SR: NAD_1983_UTM_Zone_20N
   Vertical SR: VIVD09_height



---------------- nooverlap_GBC15a_slope_p26920_v6642 ----------------: 

   Number bands = 1
   Pixel type = F32
   ⬇️ Min = 1.0411026707402016e-08
   ⬆️ Max = 73.05693817138672
   📐 Cell size = 472.845274184348
   Horizontal SR: NAD_1983_UTM_Zone_20N
   Vertical SR: VIVD09_height



---------------- USC10m_slope_p26920_v6642 ----------------: 

   Number bands = 1
   Pixel type = F32
   ⬇️ Min = 8.323395792042554e-12
   ⬆️ Max = 89.75553131103516
   📐 Cell size = 10.0
   Horizontal SR: NAD_19

In [57]:
# ── 1. CREATE NEW BATHY + SLOPE MOSAIC DATASETS ──────────────────────────────────────────────────────

# Create Mosaic Dataset (Data Management Tools): Function
def create_mosaic_dataset(md_name):

    try:
        arcpy.management.CreateMosaicDataset(
            in_workspace=gdb,
            in_mosaicdataset_name=md_name,
            coordinate_system=arcpy.SpatialReference(26920, 6642),
            # all input rasters have 1 band
            num_bands=1,
            # ** use wider pixel type of input rasters (bathy USC 32-bit float vs GBC 16-bit short; slope both 32-bit float) to avoid truncating 32-bit float decimal depth values
            pixel_type="32_BIT_FLOAT"
            # product_definition defaults to "NONE" (generic)
        )
        # define path to new mosaic for use in future blocks
        md = os.path.join(gdb, md_name)
        print(f"✅ Created mosaic dataset: {md} \n")
        # return full path to mosaic so downstream blocks can use the path without reconstructing it
        return md

    # print warning if tool failed
    except Exception as e:
        print(f"❌[FAILURE] Mosaic creation failed for {md_name}: {e} \n")
        return None


# Call function to create bathy and slope mosaic datasets + retain returned paths in variables "md_bathy" and "md_slope"
md_bathy = create_mosaic_dataset("mosaic_bathy_USCGBC_p26920_v6642")
md_slope = create_mosaic_dataset("mosaic_slope_USCGBC_p26920_v6642")

✅ Created mosaic dataset: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\MASTER_RASTERS_USC_GBC_MERGED.gdb\mosaic_bathy_USCGBC_p26920_v6642 

✅ Created mosaic dataset: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\MASTER_RASTERS_USC_GBC_MERGED.gdb\mosaic_slope_USCGBC_p26920_v6642 



In [58]:
# ── 2. ADD RASTERS TO MOSAIC DATASETS ────────────────────────────────────────────────────────────────

# define mosaic paths again so they are independent of step 1 try/except scope
md_bathy = os.path.join(gdb, "mosaic_bathy_USCGBC_p26920_v6642")
md_slope = os.path.join(gdb, "mosaic_slope_USCGBC_p26920_v6642")


# Add Rasters to Mosaic Dataset (Data Management Tools): Function
def add_rasters_to_mosaic(md, input_rasters):

    # count rasters already in mosaic before adding
    count_before = int(arcpy.management.GetCount(md)[0])

    try:
        arcpy.management.AddRastersToMosaicDataset(
            in_mosaic_dataset=md,
            raster_type="Raster Dataset",
            input_path=input_rasters,
            # "UPDATE_CELL_SIZES" (default) calculates cell size ranges for each input raster
            update_cellsize_ranges="UPDATE_CELL_SIZES",
            # "UPDATE_BOUNDARY" (default) creates/updates boundary polygon by merging all footprint polygons of input rasters into single boundary representing extent of valid pixels in mosaic
            update_boundary="UPDATE_BOUNDARY",
            # overview parameter applies to *mosaic* (overviews are lower-res images created to increase display speed/reduce CPU usage)
            update_overviews="NO_OVERVIEWS",
            # leave spatial_reference (defaults to coord system set for mosaic (same coord system as all input rasters) + sub_folder (default includes files from all subfolders within root input_path directory) blank
            # "OVERWRITE_DUPLICATES" new duplicate rasters overwrite existing rasters **matches on raster path so only overwrites if re-adding exact same file paths**
            duplicate_items_action="OVERWRITE_DUPLICATES",
            # pyramids parameter applies to *input rasters* (not mosaic itself); pyramid params default to resampling type "NEAREST" (ok here since passing raster data straight through, no reprojection/etc.); (pyramids are lower-res copy of data displayed when zoomed out and increases res as zoom in - termed "display resampling")
            build_pyramids="BUILD_PYRAMIDS",
            # "CALCULATE_STATISTICS" calculates stats for each input source raster (skips input rasters that already has stats IF skip factor value (default x=1 y=1) matches current settings)
            calculate_statistics="CALCULATE_STATISTICS"
        )
        print(f"✅ Added rasters to {os.path.basename(md)} mosaic (see below messages for details!) \n")
    # print warning if tool failed
    except Exception as e:
        print(f"❌[FAILURE] Add rasters to mosaic failed for {os.path.basename(md)}: {e} \n")

    # count rasters in mosaic after adding (*should be 2 since 2 input rasters)
    count_after = int(arcpy.management.GetCount(md)[0])
    # newly added = net increase in raster count
    added = count_after - count_before
    # overwritten = inputs that were already registered (OVERWRITE_DUPLICATES replaces in place, so no net count change); infer from count math since mosaic has no plain Path field to match against
    overwritten = len(input_rasters) - added if added < len(input_rasters) else 0
    print(f"🞥 Added new rasters = {added},    ⭮ Overwritten rasters = {overwritten},      🏱 Total rasters in mosaic = {count_after} \n")
    # total n processed rasters should = n input files (*if processed < input it means some rasters failed to add - print warning message)
    processed = added + overwritten
    if processed < len(input_rasters):
        print(f"⚠️[WARNING] {len(input_rasters) - processed} raster(s) failed to add \n")
        print(arcpy.GetMessages())


# Call function to add input rasters to bathy and slope mosaic datasets
add_rasters_to_mosaic(md_bathy, [gbc_bathy, usc_bathy])
add_rasters_to_mosaic(md_slope, [gbc_slope, usc_slope])

✅ Added rasters to mosaic_bathy_USCGBC_p26920_v6642 mosaic (see below messages for details!) 

🞥 Added new rasters = 2,    ⭮ Overwritten rasters = 0,      🏱 Total rasters in mosaic = 2 

✅ Added rasters to mosaic_slope_USCGBC_p26920_v6642 mosaic (see below messages for details!) 

🞥 Added new rasters = 2,    ⭮ Overwritten rasters = 0,      🏱 Total rasters in mosaic = 2 



In [59]:
# ── 3. EXPORT MOSAIC DATASETS AS SINGLE STANDALONE RASTERS ───────────────────────────────────────────

# Set Mosaic Dataset Properties (Data Management Tools) + Copy Raster (Data Management Tools): Function
def export_final_raster(md, final_raster, snap_raster):

    # Set Mosaic Dataset Properties (Data Management Tools):
    arcpy.management.SetMosaicDatasetProperties(
        in_mosaic_dataset=md,
        # rule that will be used for resolving overlapping pixels ("LAST" displays last image in Attribute Table)
        # ** RASTER ORDER IN MOSAIC SET BY "add_rasters_to_mosaic(md_*, [gbc_*, usc_*])" LINE IN ABOVE BLOCK (LAST RASTER = USC) **
        mosaic_operator="LAST",
        # max pixel size difference allowed before images are considered to have a different cell pixel
        # ** Copy Raster tool default will drop items whose pixel size range does NOT include requested output cell size (arcpy.env.cellSize = "10"), so increase cell size tolerance by factor of 1000 to avoid dropping GBC item whose ~450m-centered range does not include 10m requested output by increasing valid native cell size to 1000x smaller/larger than requested output cell size **
        cell_size_tolerance=1000
    )

    # ** set mosaics to export with cell size = 10 m so USC portion of final output rasters keeps native 10 m resolution (GBC portion of output rasters is resampled from native ~450 m to 10 m resolution in final output rasters only, NOT in mosaics or input rasters)
    arcpy.env.cellSize = "10"
    # use bilinear resampling (ideal method for continuous data like depth/slope) for any cells where output grid doesn't line up one-to-one with the cell in source grid (due to different cell size/offset grid origin/reprojection/etc) 
    # ** ONLY GBC CELLS GET RESAMPLED (CELL SIZE INPUT 450 M ≠ OUTPUT 10 M); USC CELLS ARE NOT RESAMPLED BECAUSE THEY ALREADY HAVE SAME CELL SIZE AS OUTPUT (10 M) + SNAP RASTER LINE BELOW ENSURES OUTPUT CELL BOUNDARIES ALIGN EXACTLY WITH INPUT (SO USC VALUES PASS THROUGH UNCHANGED TO FINAL OUTPUT RASTER) **
    arcpy.env.resamplingMethod = "BILINEAR"
    # ** snap output raster to input USC raster grid so USC cells pass through unchanged (will NOT change position of GBC raster cells)
    arcpy.env.snapRaster = snap_raster

    # Copy Raster (Data Management Tools):
    try:
        arcpy.management.CopyRaster(
            in_raster=md,
            out_rasterdataset=final_raster,
            # ** use wider pixel type of the rasters in mosaic (bathy USC 32-bit float vs GBC 16-bit short; slope both 32-bit float) to avoid truncating 32-bit float decimal depth values
            pixel_type="32_BIT_FLOAT"
        )
        print(f"✅ Exported final raster: {final_raster} \n")
    # print warning if export failed
    except Exception as e:
        print(f"❌[FAILURE] Export raster failed for {os.path.basename(final_raster)}: {e} \n")


# Call function for bathy and slope mosaic datasets
export_final_raster(md_bathy, final_bathy_raster, usc_bathy)
export_final_raster(md_slope, final_slope_raster, usc_slope)

# Clear export settings in environment for subsequent blocks
arcpy.ClearEnvironment("cellSize")
arcpy.ClearEnvironment("resamplingMethod")
arcpy.ClearEnvironment("snapRaster")

✅ Exported final raster: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\MASTER_RASTERS_USC_GBC_MERGED.gdb\BATHY_USCGBC_p26920_v6642 

✅ Exported final raster: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\MASTER_RASTERS_USC_GBC_MERGED.gdb\SLOPE_USCGBC_p26920_v6642 



In [60]:
# ── INSPECT OUTPUT MERGED RASTERS DATA ───────────────────────────────────────────────────────────────

# Calculate Statistics (Data Management Tools): (default X/Y skip factor = 1)
for stats_path2 in [final_bathy_raster, final_slope_raster]:
    arcpy.management.CalculateStatistics(in_raster_dataset=stats_path2)

# Call function to print info for each output merged raster
print_raster_info(final_bathy_raster)
print_raster_info(final_slope_raster)


---------------- BATHY_USCGBC_p26920_v6642 ----------------: 

   Number bands = 1
   Pixel type = F32
   ⬇️ Min = -8617.681640625
   ⬆️ Max = 0.0
   📐 Cell size = 10.0
   Horizontal SR: NAD_1983_UTM_Zone_20N
   Vertical SR: VIVD09_height



---------------- SLOPE_USCGBC_p26920_v6642 ----------------: 

   Number bands = 1
   Pixel type = F32
   ⬇️ Min = 5.132211322589342e-10
   ⬆️ Max = 89.68153381347656
   📐 Cell size = 10.0
   Horizontal SR: NAD_1983_UTM_Zone_20N
   Vertical SR: VIVD09_height




In [61]:
##### NOTE: IF WANT TO ADD METADATA TO OUTPUT MERGED RASTERS BEFORE EXPORTING AS TIFF FILES, DO IT NOW (IN CATALOG VIEW) #####

In [62]:
# ── 4. EXPORT FINAL MERGED RASTERS AS TIFF FILES ─────────────────────────────────────────────────────

# Copy Raster (Data Management Tools): Function
def export_tif(in_rast, tif, tif_snap_raster):

    # ** use Nearest Neighbor resampling when exporting source raster to .tif without changing grid (since snap raster to input = no resampling actually happens, so NN just guarantees pixel values pass through untouched)
    arcpy.env.resamplingMethod = "NEAREST"
    # ** snap to input raster = ensures same grid to avoid resampling
    arcpy.env.snapRaster = tif_snap_raster

    try:
        arcpy.management.CopyRaster(
            in_raster=in_rast,
            out_rasterdataset=tif,
            # ** must set NoData value when exporting as TIFF or land areas will get filled
            nodata_value=555555,
            # set to same as input raster to avoid truncating 32-bit float decimal depth/slope values
            pixel_type="32_BIT_FLOAT",
            # set output file type as TIFF
            format="TIFF"
        )
        print(f"✅ Exported TIFF file: {tif} \n")

    # print warning if export failed
    except Exception as e:
        print(f"❌[FAILURE] Export TIFF failed for {os.path.basename(tif)}: {e} \n")


# Call function for bathy and slope rasters
export_tif(final_bathy_raster, bathy_tif, final_bathy_raster)
export_tif(final_slope_raster, slope_tif, final_slope_raster)

✅ Exported TIFF file: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\exported_files\BATHY_USCGBC_p26920_v6642.tif 

✅ Exported TIFF file: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\exported_files\SLOPE_USCGBC_p26920_v6642.tif 

